# 02 — Feature Engineering
## Student 2 (ML Engineer) — Leakage-Safe Feature Pipeline

This notebook demonstrates the feature engineering process for the StockSense demand forecasting system.

**Key principles:**
- All features are **leakage-safe** — computed using only information available before the prediction window
- Uses `corrected_demand` (not raw `quantity_sold`) to avoid the censored-demand trap
- `.shift(1)` is applied before `.rolling()` to prevent today's value leaking into today's rolling average
- Sparse-history products fall back to category-level mean demand

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Add src/ to path
sys.path.insert(0, os.path.join('..', 'src'))

sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.figsize'] = (12, 6)

print('Setup complete.')

## 1. Load Master Table
Produced by Student 1 — one row per date × store × product.

In [ ]:
MASTER_TABLE_PATH = os.path.join('..', 'data', 'processed', 'master_table.csv')

df = pd.read_csv(MASTER_TABLE_PATH, parse_dates=['date'])
print(f'Master table: {len(df):,} rows × {len(df.columns)} columns')
print(f'Date range: {df["date"].min().date()} → {df["date"].max().date()}')
print(f'Stores: {df["store_id"].nunique()}, Products: {df["product_id"].nunique()}')
df.head()

## 2. Run Feature Engineering Pipeline
All feature functions are in `src/features.py` for reusability and unit-testability.

In [ ]:
from features import (
    build_features,
    add_time_features,
    add_lag_features,
    add_rolling_features,
    add_inventory_features,
    add_price_promo_features,
    create_target_variable,
)

# Run the full pipeline
df_features = build_features(df)
print(f'Feature table: {len(df_features):,} rows × {len(df_features.columns)} columns')
print(f'\nNew columns added:')
new_cols = set(df_features.columns) - set(df.columns)
for col in sorted(new_cols):
    print(f'  - {col}')

## 3. Feature Distributions
Visual sanity check — no suspicious spikes or constant values.

In [ ]:
feature_cols = ['lag_1', 'lag_7', 'lag_14', 'rolling_mean_7', 'rolling_mean_14',
                'rolling_std_7', 'days_of_inventory', 'price_change']

available = [c for c in feature_cols if c in df_features.columns]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for idx, col in enumerate(available[:8]):
    ax = axes[idx // 4][idx % 4]
    df_features[col].dropna().hist(bins=50, ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(col, fontsize=11, fontweight='bold')
    ax.set_ylabel('')

plt.suptitle('Feature Distributions — Sanity Check', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Leakage Check — Correlation with Target
Features should correlate with `next_7_day_demand` but NOT suspiciously close to 1.0 (which would signal leakage).

In [ ]:
if 'next_7_day_demand' in df_features.columns:
    numeric_cols = df_features.select_dtypes(include=[np.number]).columns.tolist()
    corr_with_target = df_features[numeric_cols].corr()['next_7_day_demand'].drop('next_7_day_demand')
    corr_sorted = corr_with_target.abs().sort_values(ascending=False).head(15)

    fig, ax = plt.subplots(figsize=(10, 6))
    corr_sorted.plot(kind='barh', ax=ax, color='teal', edgecolor='white')
    ax.set_title('Top 15 Features — Absolute Correlation with Target', fontsize=13, fontweight='bold')
    ax.set_xlabel('|Correlation|')
    ax.axvline(x=0.95, color='red', linestyle='--', alpha=0.7, label='Leakage threshold (0.95)')
    ax.legend()
    plt.tight_layout()
    plt.show()

    # Flag anything suspiciously high
    suspicious = corr_sorted[corr_sorted > 0.95]
    if len(suspicious) > 0:
        print('⚠️  POSSIBLE LEAKAGE — features with >0.95 correlation:')
        for feat, val in suspicious.items():
            print(f'   {feat}: {val:.4f}')
    else:
        print('✅ No features with suspiciously high correlation (>0.95). Leakage check passed.')

## 5. NaN Analysis
Check how many NaN values exist in each feature — expected for early rows (lag/rolling need history).

In [ ]:
nan_counts = df_features[sorted(new_cols)].isnull().sum()
nan_pct = (nan_counts / len(df_features) * 100).round(2)

nan_report = pd.DataFrame({'NaN Count': nan_counts, 'NaN %': nan_pct})
nan_report = nan_report[nan_report['NaN Count'] > 0].sort_values('NaN %', ascending=False)

if len(nan_report) > 0:
    print('Features with NaN values (expected for lag/rolling in early rows):')
    print(nan_report.to_string())
else:
    print('✅ No NaN values in any engineered feature.')

## 6. Save Feature Table

In [ ]:
output_path = os.path.join('..', 'data', 'processed', 'feature_table.csv')
df_features.to_csv(output_path, index=False)
print(f'✅ Feature table saved to: {output_path}')
print(f'   {len(df_features):,} rows × {len(df_features.columns)} columns')

---
**Next:** `03_demand_forecasting.ipynb` — train and evaluate the demand model using this feature table.